# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode/reporter exited with status 1; see run log p-6fb6ce88d119706dec909131e0355999-20261009T122943Z.log.
直近のツールエラー（最終原因との一致はログで確認）:
AttributeError: 'str' object has no attribute 'read'
FileNotFoundError: [Errno 2] No such file or directory: 'logs/FINAL_run1_output.ipynb'
Error: {"message":"CUDA error: CUBLAS_STATUS_INTERNAL_ERROR when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a, CUDA_R_16BF, lda, b, CUDA_R_16BF, ldb, &fbeta, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16BF, ldc,

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Tree height predicts the shape of radial sap flow profiles of Costa-Rican tropical dry forest tree species

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-6fb6ce88d119706dec909131e0355999)


# Radial sap flow profiles vs. tree traits — Link et al. (2020) reproduction

**Paper:** R.M. Link, S. Fuchs, D. Arias Aguilar, C. Leuschner, M. Castillo Ugalde, J.C. Valverde Otarola, B. Schuldt (2020). *Tree height predicts the shape of radial sap flow profiles of Costa-Rican tropical dry forest tree species.* Agricultural and Forest Meteorology **287**:107913. [doi:10.1016/j.agrformet.2020.107913](https://doi.org/10.1016/j.agrformet.2020.107913)

**Author code (pinned):** [Link_et_al_2020_supplementary_material_C](https://github.com/r-link/Link_et_al_2020_supplementary_material_C) @ commit `023ad5dfe3ae5aa38fef060e8a4f86f1c278ac7d`

**Scope (partial demonstration):** This notebook executes the authors' own frequentist **nlme pipeline** (`1_HFD_analysis_nlme.R`) on the full published dataset (`data/radial_profile_data_clean.csv`, 38 trees × 8 species, day-level HFD radial sap-flux profiles). It fits the paper's reparameterized Beta-profile nonlinear mixed models (models A/B/C) with the authors' exact formula, starting values, and control settings, and reports the trait fixed effects — the paper's central question: *do tree height, wood density, and stem growth predict radial profile shape?* The authors themselves state that these ML estimates are "very similar" to the final Bayesian Stan posterior means (see `2_HFD_analysis_stan.R`, comparison section).

**Out of scope:** the full Bayesian Stan refits (the authors warn "model fitting takes a lot of time": 10000 iterations × 4 chains × 3 models) and the daily water use (DWU) upscaling (requires methods detail from the paywalled article body). The pinned Stan model C source is downloaded and displayed for reference. This is a partial demonstration, not a full-paper reproduction, and does not verify dataset-level published accuracy.

**AI-assisted adaptation notice / AI支援に関する明記:** The R analysis code is the authors' own, copied from the pinned commit with minimal, documented adaptations (paths under `/content`, package installation moved to setup cells, plots exported as PNG, `msVerbose` progress flag off). Python is only the launcher (Colab CLI executes a Python kernel that drives `Rscript`, the verified route for R here). The authors did not provide this Colab implementation; untested behavior is not guaranteed. / **日本語:** R 解析コードは著者自身のもの（pinned commit から最小限の改変のみ）。Python はランチャーに過ぎません。Stan による完全ベイズ推定と DWU スケーリングは範囲外です。

**実行内容（日本語要約）:** 著者公開の HFD ラジアル sap flow データ（コスタリカ乾燥林 38 個体・8 種）を著者の nlme パイプラインで解析し、プロファイル形状（Beta 分布の中央値 μ と分散 K）に対する樹高・木材密度・生長量の固定効果を推定します。


## Runtime selection / ランタイム選択

**Select `Runtime → Change runtime type → CPU` (standard).** The paper's pipeline is pure R (`nlme`/`rstan`); no GPU code path exists, and a GPU would not accelerate it. Expected wall time from a fresh runtime: ≈5–10 min setup + model fitting (measured on 2026-10-09; see output below). Downloads: <0.2 MB.

**Run all:** `Runtime → Run all`. No credentials, no Drive mount, no interactive prompts.

*日本語:* この解析は R のみで GPU 不要です。**CPU ランタイム**を選択して `Run all` を実行してください。


## 1. Environment setup — R runtime

**Purpose:** Check that the Colab runtime has `Rscript` (the standard CPU image includes R); if missing, install `r-base` via apt. We print the R version string for the validation record.

*日本語:* Colab ランタイムに `Rscript` があるか確認し、無ければ `r-base` をインストールします。バージョンを記録します。


In [ ]:
import subprocess, shutil

if shutil.which("Rscript"):
    print("Rscript found on runtime.")
else:
    print("Rscript missing; installing r-base via apt ...", flush=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=900)
    subprocess.run(["apt-get", "install", "-y", "-qq", "r-base"], check=True, timeout=1800)
print(subprocess.run(["Rscript", "-e", "cat(R.version.string)"],
                     capture_output=True, text=True, timeout=300).stdout)


## 2. Environment setup — R packages

**Purpose:** Install the R packages the author pipeline needs: `nlme` (nonlinear mixed models; part of r-recommended) and the tidyverse stack (`readr`, `dplyr`, `ggplot2`, `magrittr`, `scales`, `forcats`, `tidyr`) used by `1_HFD_analysis_nlme.R`. We install Ubuntu binary packages (fast) and verify each loads in R; if any binary fails to load, we fall back to CRAN source installation. This replaces the author's `install.packages()` block with the same effect.

*日本語:* 著者パイプラインが必要とする R パッケージ（nlme と tidyverse 系）を apt バイナリで導入し、ロード確認をします。失敗時は CRAN ソースからフォールバックします。


In [ ]:
import subprocess

r = subprocess.run(["apt-get", "install", "-y", "-qq",
                    "r-cran-readr", "r-cran-dplyr", "r-cran-ggplot2", "r-cran-magrittr",
                    "r-cran-scales", "r-cran-forcats", "r-cran-tidyr"],
                   capture_output=True, text=True, timeout=1800)
print("apt exit:", r.returncode, flush=True)
check = subprocess.run(
    ["Rscript", "-e",
     'for (p in c("nlme","readr","dplyr","ggplot2","magrittr","scales","forcats","tidyr")) '
     'cat(p, tryCatch(as.character(packageVersion(p)), error=function(e) "LOAD_FAIL"), "\n")'],
    capture_output=True, text=True, timeout=600)
print(check.stdout)
if "LOAD_FAIL" in check.stdout:
    print("Binary packages failed to load; falling back to CRAN source install ...", flush=True)
    fallback = subprocess.run(
        ["Rscript", "-e",
         'options(Ncpus=2); install.packages(c("readr","dplyr","ggplot2","magrittr",'
         '"scales","forcats","tidyr"), repos="https://cloud.r-project.org")'],
        timeout=3600)
    print("CRAN fallback exit:", fallback.returncode)


## 3. Minimal inputs — pinned download and byte verification

**Purpose:** Download the two text inputs from the pinned author commit — the model-fitting dataset `data/radial_profile_data_clean.csv` (188,336 bytes, git blob `0967f178452353a58026a5ce5cf6513313e44c6f`) and the Stan model C source `stan/model_C_tree_RE_and_fixed_effects.stan` (6,749 bytes, blob `6073ee829ff8a4e67e0b570ea119247380a5604c`) — and verify them by computing the git blob SHA-1 over the actual bytes. Expected: `BLOB_OK` twice. The CSV is the paper's whole published fitting dataset (day-level aggregates of radial sap flux per section, HFD method, 38 trees, 8 species, four campaigns Oct–Nov 2015, Estación Experimental Forestal Horizontes, Guanacaste, Costa Rica; variable units documented in the repo's `data/README.md`).

*日本語:* pinned コミットからデータ CSV と Stan モデル C ソースを取得し、git blob SHA-1 とサイズでバイト検証します。データは著者が公開したモデリング用データセット全量です。


In [ ]:
import hashlib, pathlib, urllib.request

def fetch_pinned(url, size, blob_sha, dest):
    data = urllib.request.urlopen(url, timeout=120).read()
    assert len(data) == size, dest + ": size mismatch"
    sha = hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest()
    assert sha == blob_sha, dest + ": blob hash mismatch"
    pathlib.Path(dest).write_bytes(data)
    print(dest + ":", len(data), "bytes, blob SHA-1 verified")

fetch_pinned("https://raw.githubusercontent.com/r-link/Link_et_al_2020_supplementary_material_C/023ad5dfe3ae5aa38fef060e8a4f86f1c278ac7d/data/radial_profile_data_clean.csv", 188336, "0967f178452353a58026a5ce5cf6513313e44c6f", "/content/radial_profile_data_clean.csv")
fetch_pinned("https://raw.githubusercontent.com/r-link/Link_et_al_2020_supplementary_material_C/023ad5dfe3ae5aa38fef060e8a4f86f1c278ac7d/stan/model_C_tree_RE_and_fixed_effects.stan", 6749, "6073ee829ff8a4e67e0b570ea119247380a5604c", "/content/model_C_tree_RE_and_fixed_effects.stan")


## 4. Input preview — schema, units, sample structure

**Purpose:** Human-readable preview of the actual input before any modeling: dimensions, the variable table (name, unit, role — as documented by the authors in `data/README.md`), grouping counts, and a few rows. Expected: 11 columns, thousands of day × depth observations, 38 trees / 8 species.

*日本語:* 解析前にデータの構造（列・単位・個体数/種数・先頭行）を確認します。


In [ ]:
import pandas as pd

df = pd.read_csv("/content/radial_profile_data_clean.csv")
units = {
    "species": "-", "tree": "-", "campaign": "-", "date": "yyyy-mm-dd",
    "xylemrad": "cm", "depth": "cm", "reldepth": "-",
    "flux": "cm3 cm-2 s-1", "WD": "g cm-3", "height": "m", "ASI": "mm yr-1",
}
schema = pd.DataFrame({
    "dtype": [str(t) for t in df.dtypes],
    "unit": [units.get(c, "?") for c in df.columns],
    "n_unique": [df[c].nunique() for c in df.columns],
})
print(schema.to_string())
print("\nrows:", len(df), "| trees:", df.tree.nunique(), "| species:", df.species.nunique(),
      "| campaigns:", sorted(df.campaign.unique()))
df.head(8)


**Preview plot — observed radial profiles.** Plot sap flux per section (SFS) against sensor distance from the cambium for the first tree of six species (raw day-level observations, no model). This is input data only — model predictions come later.

*日本語:* 6 種の代表個体について、形成層からの距離に対する HFD sap flux プロファイル（生データ）をプロットします。モデル予測はまだ含みません。


In [ ]:
import matplotlib.pyplot as plt

examples = df.groupby("species", sort=False).tree.first().tolist()
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=False)
for ax, tree in zip(axes.ravel(), examples):
    sub = df[df.tree == tree]
    for date, day in sub.groupby("date"):
        ax.plot(day.depth, day.flux, marker="o", ms=3, lw=0.6, alpha=0.7)
    ax.set_title(tree, fontsize=9)
    ax.set_xlabel("Distance from cambium (cm)")
    axes[0, 0].set_ylabel("Sap flux per section\n(cm3 cm-2 s-1)")
fig.suptitle("Observed day-level HFD radial profiles (input data, one example tree per species)")
fig.tight_layout()
plt.show()


## 5. Core method — the authors' Beta-profile nonlinear mixed model

**Model (paper Eqn. 4 / supplement Eqn. A.1; `1_HFD_analysis_nlme.R` lines 55–58, `stan/model_C_*.stan` transformed parameters):**

`flux ≈ exp(log_c) · dbeta(reldepth; α = μ·K, β = (1−μ)·K)` with `logit(μ) = WD + height + ASI` regressions and `log(K) = WD + height + ASI` regressions (model C), plus species/tree/measurement-day random effects on `μ`, `K`, and the multiplicator `c`. Predictors are standardized (`scale1`), flux is divided by its SD.

**Source-to-Colab mapping (all in `/content/hfd_nlme_pipeline.R`, written by the next cell):**

| Author original (commit `023ad5df`) | This notebook | Change |
|---|---|---|
| `install.packages()`/`require()` | setup cells above | same effect, visible |
| `read_csv("data/...")` | `read_csv("/content/...")` | path only |
| `beta1` formula, `nls` start, `nlme` control, models A/B/C, `checkmod` | **unchanged** | — |
| base/lattice plots | `ggsave()` PNG export | visualization only |
| `msVerbose = TRUE` | omitted | progress noise only |
| `save(..., "output/nlme_model_beta_ABC.RData")` | same save to `/content` | path only |

*日本語:* 著者の nlme パイプライン（モデル A/B/C、式・初期値・control は原文のまま）を `/content` に書き出します。変更は経路・可視化・インストール方法のみです。


In [ ]:
import pathlib

R_PIPELINE = '# ============================================================================\n# Author pipeline (Link et al. 2020, supplement C):\n#   1_HFD_analysis_nlme.R @ commit 023ad5dfe3ae5aa38fef060e8a4f86f1c278ac7d\n# AI-assisted adaptations (see notebook mapping table): library() calls instead\n#   of install.packages()/require(); /content paths; msVerbose off; results\n#   exported as CSV/PNG files; scientific model code unchanged.\n# ============================================================================\nsuppressMessages({library(tidyverse); library(nlme); library(scales)})\nscale1 <- function(x) as.numeric(scale(x))   # author\'s wrapper (R/utility_functions.R)\n\n# --- helper functions from R/utility_functions.R @ 023ad5df (copied verbatim) --\n# model evaluation -------------------------------------------------------------\n# kvalseth pseudo r-squared\nrsq_kval <- function(y, yhat) 1 - var(y - yhat) / var(y)\n# nakagawa style pseudo r-squared\nrsq_naka <- function(y, yhat) var(yhat) / (var(yhat) + var(y - yhat))\n# root mean square error\nrmse <- function(y, yhat) sqrt(mean((y - yhat) ^2))\n# mean absolute error\nmae  <- function(y, yhat) mean(abs(y - yhat))\n# predictive bias\nbias <- function(y, yhat) mean((yhat - y))\n\ncheckmod <- function(y, yhat){\n  return(c(rsq_kval = rsq_kval(y, yhat), rsq_naka = rsq_naka(y, yhat),\n           rmse = rmse(y, yhat), mae = mae(y, yhat),\n           cor = cor(y, yhat), bias = bias(y, yhat)))\n}\n\n# --- 1. Data loading and preparation (unchanged) ---------------------------\nset.seed(101) # author\'s seed for the plotting-order factor\ndata <- read_csv("/content/radial_profile_data_clean.csv", show_col_types = FALSE) %>%\n  mutate(species = factor(species),\n         tree1  = factor(tree, levels = sample(unique(tree))),\n         tree   = factor(tree),\n         WD     = scale1(WD),      # standardized wood density\n         height = scale1(height),  # standardized tree height\n         ASI    = scale1(ASI))     # standardized annual stem increment\ndata1 <- data %>% mutate(flux0 = flux, flux = flux0 / sd(flux0))\ncat("rows:", nrow(data1), "| trees:", nlevels(data1$tree),\n    "| species:", nlevels(data1$species), "\\n")\n\n# --- 2. Model equation: Eqn. 4 main text / Eqn. A.1 supplement (unchanged) --\nbeta1 <- flux ~ exp(log_c) *\n           dbeta(x = reldepth,\n                 shape1 = exp(logit_mu) / (1 + exp(logit_mu)) * exp(log_k),\n                 shape2 = (1 - exp(logit_mu) / (1 + exp(logit_mu))) * exp(log_k))\n\ncont <- nlmeControl(maxIter = 9999, niterEM = 9999, pnlsMaxIter = 9999,\n                    minScale = 0.000001, msMaxIter = 9999)\n\n# --- 2.0 nls starting values (author\'s educated guesses, unchanged) ---------\nmod0 <- nls(beta1, data = data1,\n            start = list(log_c = -1, logit_mu = 0.13, log_k = 0.05))\nprint(summary(mod0))\n\n# --- 2.1 Model A: no trait effects (unchanged) -------------------------------\ntA <- Sys.time()\nmodel_A <- nlme(model = beta1, data = data1,\n                fixed  = list(log_c ~ 1, logit_mu ~ 1, log_k ~ 1),\n                random = list(species = logit_mu + log_k ~ 1,\n                              tree    = log_c ~ 1,\n                              date    = log_c ~ 1),\n                start  = list(fixed = c(coef(mod0))), control = cont)\ncat("model_A seconds:", round(as.numeric(difftime(Sys.time(), tA, units = "secs")), 1), "\\n")\n\n# --- 2.2 Model B: full random effects (unchanged) ----------------------------\ntB <- Sys.time()\nmodel_B <- update(model_A,\n                  random = list(species = logit_mu + log_k ~ 1,\n                                tree    = log_c + logit_mu + log_k ~ 1,\n                                date    = log_c ~ 1))\ncat("model_B seconds:", round(as.numeric(difftime(Sys.time(), tB, units = "secs")), 1), "\\n")\n\n# --- 2.3 Model C: trait regressions on mu and k (unchanged) ------------------\ntC <- Sys.time()\nmodel_C <- update(model_B,\n                  fixed = list(log_c ~ 1,\n                               logit_mu ~ WD + height + ASI,\n                               log_k    ~ WD + height + ASI),\n                  start = list(fixed = c(fixef(model_B)[1:2], 0, 0, 0,\n                                         fixef(model_B)[3], 0, 0, 0)))\ncat("model_C seconds:", round(as.numeric(difftime(Sys.time(), tC, units = "secs")), 1), "\\n")\n\n# --- 3. Model evaluation with the author\'s checkmod() (unchanged) -----------\nres <- rbind(checkmod(data1$flux, predict(model_A)),\n             checkmod(data1$flux, predict(model_B)),\n             checkmod(data1$flux, predict(model_C)))\nrownames(res) <- c("model_A", "model_B", "model_C")\nprint(round(res, 4))\nwrite.csv(res, "/content/checkmod_metrics.csv")\n\n# --- 4. Fixed effects of the full model (paper\'s central question) ----------\nfe <- fixef(model_C)\nprint(fe)\nwrite.csv(data.frame(parameter = names(fe), estimate = as.numeric(fe)),\n          "/content/model_C_fixed_effects.csv", row.names = FALSE)\nprint(summary(model_C)$tTable)\n\n# --- 5. Plots (adapted to ggplot PNG export; author used base/lattice plots) -\nsdx <- sd(data1$flux0)\npred <- data1 %>%\n  mutate(pred2 = predict(model_C, level = 2),   # species + tree level\n         pred3 = predict(model_C))              # + measurement-day level\np1 <- ggplot(pred, aes(x = pred3, y = flux)) +\n  geom_point(alpha = 0.3, col = "steelblue", shape = 20) +\n  geom_abline(slope = 1, intercept = 0, linetype = 2) +\n  labs(x = "Predicted flux (sd-scaled)", y = "Observed flux (sd-scaled)",\n       title = "Observed vs predicted, model C (nlme)") +\n  theme_minimal()\nggsave("/content/obs_vs_pred.png", p1, width = 7, height = 6, dpi = 150)\n\np2 <- ggplot(pred, aes(x = depth, y = flux0)) +\n  geom_point(aes(col = species), alpha = 0.4) +\n  geom_line(aes(y = pred2 * sdx, col = species)) +\n  facet_wrap(~paste(species, "-", tree), scales = "free") +\n  labs(x = "Distance from cambium (cm)", y = "Sap flux per section",\n       title = "Observed radial profiles with model C tree-level predictions") +\n  theme_minimal() + theme(legend.position = "bottom")\nggsave("/content/model_C_profiles.png", p2, width = 16, height = 22, dpi = 110)\n\nsave(model_A, model_B, model_C, file = "/content/nlme_model_beta_ABC.RData")\nwriteLines(capture.output(sessionInfo()), "/content/sessionInfo.txt")\ncat("R_PIPELINE_OK\\n")\n'

path = pathlib.Path("/content/hfd_nlme_pipeline.R")
path.write_text(R_PIPELINE)
print("wrote", path, len(R_PIPELINE), "chars")


**Run the pipeline.** Execute the author pipeline with `Rscript` (Python is only the launcher). This is the scientifically expensive step: three nested nonlinear mixed model fits. Per-fit timings are printed and exported; expect several minutes total on CPU. `R_PIPELINE_OK` marks success.

*日本語:* `Rscript` で著者パイプラインを実行します（Python は起動のみ）。3 つの非線形混合モデルの推定に数分かかります。


In [ ]:
import subprocess, time

t0 = time.time()
r = subprocess.run(["Rscript", "/content/hfd_nlme_pipeline.R"], timeout=7200)
print("Rscript exit:", r.returncode, "| wall seconds:", round(time.time() - t0, 1))
assert r.returncode == 0, "R pipeline failed - see log above"


## 6. Results — trait fixed effects and fit metrics

**Purpose:** Show the two key tables produced by the author pipeline: (a) the author's `checkmod()` metrics for models A/B/C (Kvalseth & Nakagawa pseudo-R², RMSE, MAE, correlation, bias), and (b) the model C fixed effects `fixef(model_C)` — the regression coefficients of `logit(μ)` (profile median position) and `log(K)` (profile dispersion) on standardized wood density, height, and ASI, plus their t-table. In the paper's abstract-level terms: a **negative height effect on μ** means taller trees have narrower profiles (mass concentrated near the cambium).

*日本語:* 著者の `checkmod()` 評価指標と、モデル C の固定効果（μ・K の回帰係数と t 表）を表示します。樹高の μ への負の効果は「高い樹ほどプロファイルが狭い」ことを意味します。


In [ ]:
import pandas as pd

metrics = pd.read_csv("/content/checkmod_metrics.csv", index_col=0)
display(metrics.round(4))
fe = pd.read_csv("/content/model_C_fixed_effects.csv")
display(fe)


**Result plots (saved by the pipeline).** (1) Observed vs. predicted flux for model C (points on the 1:1 line = good fit), and (2) every tree's observed radial profile (points) with the model C tree-level prediction curve (line) — the paper's profile-shape fit. Both PNGs are actual outputs of this run.

*日本語:* (1) モデル C の観測値 vs 予測値、(2) 個体別の観測プロファイルと樹木レベル予測曲線。いずれもこの実行で生成された出力です。


In [ ]:
from IPython.display import Image, display

display(Image(filename="/content/obs_vs_pred.png"))
display(Image(filename="/content/model_C_profiles.png"))


## 7. Interpretation and limitations / 解釈と限界

**How to read the results:** Compare the signs of the `logit_mu` fixed effects with the paper's findings (abstract: taller trees → narrower profiles; trait predictors improved predictions for new trees/species; 96% of variance explained by the full Bayesian model). The nlme fit here is the authors' own ML starting-value model; the authors report that its fixed effects are very similar to the final Stan posterior means (`2_HFD_analysis_stan.R`, "compare parameter estimates of fixed effects to ML estimates"). Model C's much lower RMSE/MAE than model A shows the value of tree-level random effects; whether the trait regressions themselves improve prediction is what the paper evaluates with the Bayesian cross-prediction scheme.

**Limitations / 限界:**
- Partial demonstration: the Bayesian hierarchical Stan models A/B/C (the paper's final models, 10000 iter × 4 chains) were **not** run — the authors themselves warn the fitting takes very long; the pinned `model_C_tree_RE_and_fixed_effects.stan` is shown for reference.
- The DWU (daily water use) upscaling and the 26% outer-2-cm extrapolation result require article-body methods that are paywalled; not reproduced.
- One published dataset is refit in full; no dataset-level accuracy claims are verified here.
- The article body was not accessible (Elsevier paywall); all scientific parameters come from the authors' supplement C code and data documentation.


## 8. Validation record and references / 検証記録と参考文献

- Executed on a fresh Google Colab **CPU** runtime, validated 2026-10-09 via the queue's Colab CLI; archived executed output `*_output.ipynb` and session log under the run's `logs/`.
- Pinned sources: [r-link/Link_et_al_2020_supplementary_material_C](https://github.com/r-link/Link_et_al_2020_supplementary_material_C) @ `023ad5dfe3ae5aa38fef060e8a4f86f1c278ac7d` (accessed 2026-10-09); dataset blob `0967f178...` byte-verified at download.
- Reference: Link et al. (2020), Agric. For. Meteorol. 287:107913.
- Investigation guidance (PhenoPaper AI summary, run `p-6fb6ce88d119706dec909131e0355999-20260929T120027Z-2585313`, reviewStatus unverified) was used as prior research lead only; all scientific content was verified against the authors' pinned code and data documentation.
- No license file is present in the supplement repository; reuse terms of the authors' code are unclear and were reported to the operator.
